# Daily Price Update — v2
Fetches NSE stock OHLC data via yfinance and upserts into SQL Server.

**Key improvements over v1:**
- Main loop is now a **code cell** (was accidentally a markdown cell)
- Batch downloads using `yf.download()` — far fewer API calls
- Exponential backoff on rate-limit errors (HTTP 429 / `TooManyRequests`)
- Jittered sleep between batches to avoid detection
- Upsert pattern (skip duplicates) so re-runs are safe

In [1]:
import yfinance as yf
import pandas as pd
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta
import time
import random
import logging

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

C:\Users\schak\AppData\Roaming\Python\Python39\site-packages\pandas\core\computation\expressions.py:21: UserWarning: Pandas requires version '2.8.4' or newer of 'numexpr' (version '2.8.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
C:\Users\schak\AppData\Roaming\Python\Python39\site-packages\pandas\core\arrays\masked.py:61: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [2]:
# ──────────────────────────────────────────
#  CONFIG
# ──────────────────────────────────────────

DB_URL = (
    "mssql+pyodbc://@localhost\\SQLEXPRESS/StockAnalystTrack"
    "?driver=ODBC+Driver+17+for+SQL+Server"
    "&trusted_connection=yes"
)

DEFAULT_LOOKBACK = 365   # days back if symbol has no existing data

BATCH_SIZE   = 20        # symbols per yf.download() call  ← tune this (10–25 is safe)
SLEEP_MIN    = 3.0       # min seconds between batches
SLEEP_MAX    = 6.0       # max seconds between batches  (random jitter)
MAX_RETRY    = 5         # retries per batch on failure
BACKOFF_BASE = 15        # seconds; doubles each retry

# ──────────────────────────────────────────
#  DB ENGINE
# ──────────────────────────────────────────

engine = create_engine(DB_URL, fast_executemany=True)

In [3]:
# ──────────────────────────────────────────
#  HELPERS
# ──────────────────────────────────────────

def chunk(lst, n):
    """Split list into chunks of size n."""
    for i in range(0, len(lst), n):
        yield lst[i : i + n]


def get_symbols():
    with engine.connect() as conn:
        df = pd.read_sql("""
            SELECT DISTINCT nse_ticker_yfinance
            FROM nseticker
            WHERE nse_ticker_yfinance IS NOT NULL
        """, conn)
    return df["nse_ticker_yfinance"].tolist()


def get_last_dates():
    with engine.connect() as conn:
        df = pd.read_sql("""
            SELECT symbol, MAX(date) AS last_date
            FROM dailyohlc
            GROUP BY symbol
        """, conn)
    df["last_date"] = pd.to_datetime(df["last_date"])
    return dict(zip(df["symbol"], df["last_date"]))


def jitter_sleep():
    t = random.uniform(SLEEP_MIN, SLEEP_MAX)
    time.sleep(t)


def is_rate_limit_error(e: Exception) -> bool:
    """Detect yfinance / requests rate-limit signals."""
    msg = str(e).lower()
    return any(kw in msg for kw in ["429", "too many requests", "rate limit", "throttl"])


def fetch_batch(symbols: list, start: str, end: str) -> pd.DataFrame:
    """
    Download a batch of symbols in one yf.download() call.
    Returns a tidy DataFrame with columns: symbol, date, open, high, low, close, volume, adjclose
    """
    tickers_str = " ".join(symbols)

    for attempt in range(MAX_RETRY):
        try:
            raw = yf.download(
                tickers_str,
                start=start,
                end=end,
                auto_adjust=False,
                group_by="ticker",
                threads=True,
                progress=False,
            )

            if raw.empty:
                return pd.DataFrame()

            # ── Normalise multi-level columns ──────────────────────────────
            records = []
            for sym in symbols:
                try:
                    # Single-ticker download has flat columns; multi has MultiIndex
                    if isinstance(raw.columns, pd.MultiIndex):
                        df = raw[sym].copy()
                    else:
                        df = raw.copy()   # only one ticker was requested

                    df = df.dropna(subset=["Close"])
                    if df.empty:
                        continue

                    df.index = pd.to_datetime(df.index).tz_localize(None)
                    df = df.rename(columns=str.lower)

                    # yfinance sometimes names it 'adj close'
                    if "adj close" in df.columns:
                        df = df.rename(columns={"adj close": "adjclose"})

                    df["symbol"] = sym
                    df["date"]   = df.index
                    df["volume"] = df["volume"].fillna(0).astype(int)

                    records.append(df[["symbol","date","open","high","low","close","volume","adjclose"]])

                except Exception as inner:
                    logging.warning(f"   ⚠ Could not parse {sym}: {inner}")

            return pd.concat(records, ignore_index=True) if records else pd.DataFrame()

        except Exception as e:
            wait = BACKOFF_BASE * (2 ** attempt)   # 15, 30, 60, 120, 240 s
            if is_rate_limit_error(e):
                logging.warning(f"   🚦 Rate-limit hit. Sleeping {wait}s before retry {attempt+1}/{MAX_RETRY}")
            else:
                logging.warning(f"   ⏳ Error: {e}. Retry {attempt+1}/{MAX_RETRY} in {wait}s")

            if attempt < MAX_RETRY - 1:
                time.sleep(wait)
            else:
                raise

In [4]:
# ──────────────────────────────────────────
#  PREP
# ──────────────────────────────────────────

today      = pd.Timestamp.today().normalize()   # midnight today
symbols    = get_symbols()
last_dates = get_last_dates()

print(f"📊 Total symbols : {len(symbols)}")
print(f"📅 Today         : {today.date()}")

# Build per-symbol start dates, skip already up-to-date
work = {}   # sym -> start date
for sym in symbols:
    if sym in last_dates and pd.notna(last_dates[sym]):
        start = last_dates[sym] + timedelta(days=1)
    else:
        start = today - timedelta(days=DEFAULT_LOOKBACK)

    if start <= today:
        work[sym] = start

print(f"🔄 Symbols needing update: {len(work)}")

📊 Total symbols : 213
📅 Today         : 2026-02-28
🔄 Symbols needing update: 213


In [5]:
# ──────────────────────────────────────────
#  MAIN LOOP  (now a CODE cell!)
# ──────────────────────────────────────────

# Group symbols by their start date so we can batch them efficiently.
# Symbols that all need the same start date can share one download call.
from collections import defaultdict

date_groups = defaultdict(list)
for sym, start in work.items():
    date_groups[start.date()].append(sym)

inserted = 0
failed   = []
end_s    = today.strftime("%Y-%m-%d")

total_batches = sum(len(syms) // BATCH_SIZE + (1 if len(syms) % BATCH_SIZE else 0)
                    for syms in date_groups.values())
batch_num = 0

with engine.begin() as conn:
    for start_date, syms in sorted(date_groups.items()):
        start_s = start_date.strftime("%Y-%m-%d")

        for batch in chunk(syms, BATCH_SIZE):
            batch_num += 1
            print(f"\n[Batch {batch_num}/{total_batches}] {len(batch)} symbols | {start_s} → {end_s}")
            print("  ", batch)

            try:
                df = fetch_batch(batch, start_s, end_s)

                if df.empty:
                    print("   ⚠ No data returned for this batch")
                    failed.extend(batch)
                    continue

                rows = df.to_dict(orient="records")

                # ── UPSERT: skip rows that already exist ──────────────────
                conn.execute(
                    text("""
                        INSERT INTO dailyohlc
                            (symbol, [date], [open], high, low, [close], volume, adjclose)
                        SELECT
                            :symbol, :date, :open, :high, :low, :close, :volume, :adjclose
                        WHERE NOT EXISTS (
                            SELECT 1 FROM dailyohlc
                            WHERE symbol = :symbol AND [date] = :date
                        )
                    """),
                    rows
                )

                inserted += len(rows)
                print(f"   ✅ Inserted up to {len(rows)} rows (dupes skipped)")

            except Exception as e:
                print(f"   ❌ Batch failed: {e}")
                failed.extend(batch)

            jitter_sleep()   # polite pause between batches


# ── SUMMARY ──────────────────────────────
print("\n" + "=" * 40)
print("SUMMARY")
print("=" * 40)
print(f"Rows inserted (gross) : {inserted}")
print(f"Failed symbols        : {len(failed)}")
if failed:
    print("Failed:", failed)
print("=" * 40)


[Batch 1/12] 2 symbols | 2025-02-28 → 2026-02-28
   ['CEAT.NS', 'TATAMOTORS.NS']


2026-02-28 14:43:33,699 ERROR 
2 Failed downloads:
2026-02-28 14:43:33,699 ERROR ['TATAMOTORS.NS', 'CEAT.NS']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


   ⚠ No data returned for this batch

[Batch 2/12] 20 symbols | 2026-02-21 → 2026-02-28
   ['360ONE.NS', 'AARTIIND.NS', 'AAVAS.NS', 'ABREL.NS', 'ACC.NS', 'ADANIPORTS.NS', 'AGARWALEYE.NS', 'AJANTPHARM.NS', 'AJAXENGG.NS', 'AKUMS.NS', 'ALKEM.NS', 'ALKYLAMINE.NS', 'AMBUJACEM.NS', 'ANANDRATHI.NS', 'ANGELONE.NS', 'APARINDS.NS', 'APLAPOLLO.NS', 'APLLTD.NS', 'APOLLOTYRE.NS', 'ARVINDFASN.NS']


2026-02-28 14:43:35,744 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:43:35,844 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:43:35,844 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:43:37,747 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:43:37,766 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:43:37,770 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:43:39,677 ERROR 
20 Failed downloads:
2026-02-28 14:43:39,677 ERROR ['APLLTD.NS', 'ANANDRATHI.NS', 'AGARWALEYE.NS', 'ALKEM.NS', 'ALKYLAMINE.NS', 'APLAPOLLO.NS', 'AARTIIND.NS', 'AJAXENGG.NS', 'ARVINDFA

   ⚠ No data returned for this batch

[Batch 3/12] 20 symbols | 2026-02-21 → 2026-02-28
   ['ASTRAL.NS', 'ASTRAMICRO.NS', 'AUBANK.NS', 'AXISBANK.NS', 'AZAD.NS', 'BAJAJ-AUTO.NS', 'BAJAJCON.NS', 'BANDHANBNK.NS', 'BANKBARODA.NS', 'BANSALWIRE.NS', 'BECTORFOOD.NS', 'BEL.NS', 'BHEL.NS', 'BIKAJI.NS', 'BLUEJET.NS', 'BPCL.NS', 'BRIGADE.NS', 'CANFINHOME.NS', 'CANHLIFE.NS', 'CASTROLIND.NS']


...

   ⚠ No data returned for this batch

[Batch 11/12] 20 symbols | 2026-02-21 → 2026-02-28
   ['TATATECH.NS', 'TATVA.NS', 'TBOTEK.NS', 'TCIEXP.NS', 'TCS.NS', 'TEAMLEASE.NS', 'TECHM.NS', 'THELEELA.NS', 'THERMAX.NS', 'TIINDIA.NS', 'TITAN.NS', 'TORNTPHARM.NS', 'TRAVELFOOD.NS', 'TRENT.NS', 'TRITURBINE.NS', 'TVSMOTOR.NS', 'UJJIVANSFB.NS', 'ULTRACEMCO.NS', 'UNIONBANK.NS', 'UNITDSPR.NS']


2026-02-28 14:44:23,140 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:44:23,375 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:44:23,379 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:44:23,380 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:44:23,381 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:44:23,382 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:44:24,836 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:44:27,462 ERROR 
20 Failed downloads:
2026-02-28 14:44:27,462 ERROR [

   ⚠ No data returned for this batch

[Batch 12/12] 11 symbols | 2026-02-21 → 2026-02-28
   ['VEDL.NS', 'VIKRAMSOLR.NS', 'VMART.NS', 'VMM.NS', 'VOLTAS.NS', 'WABAG.NS', 'WESTLIFE.NS', 'WIPRO.NS', 'YESBANK.NS', 'ZEEL.NS', 'ZYDUSWELL.NS']


2026-02-28 14:44:28,508 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:44:30,321 WARNING Connection pool is full, discarding connection: query2.finance.yahoo.com. Connection pool size: 10
2026-02-28 14:44:30,332 ERROR 
11 Failed downloads:
2026-02-28 14:44:30,333 ERROR ['WABAG.NS', 'VOLTAS.NS', 'WIPRO.NS', 'YESBANK.NS', 'VMART.NS', 'VEDL.NS', 'WESTLIFE.NS', 'ZEEL.NS', 'ZYDUSWELL.NS', 'VIKRAMSOLR.NS', 'VMM.NS']: YFRateLimitError('Too Many Requests. Rate limited. Try after a while.')


   ⚠ No data returned for this batch

SUMMARY
Rows inserted (gross) : 0
Failed symbols        : 213
Failed: ['CEAT.NS', 'TATAMOTORS.NS', '360ONE.NS', 'AARTIIND.NS', 'AAVAS.NS', 'ABREL.NS', 'ACC.NS', 'ADANIPORTS.NS', 'AGARWALEYE.NS', 'AJANTPHARM.NS', 'AJAXENGG.NS', 'AKUMS.NS', 'ALKEM.NS', 'ALKYLAMINE.NS', 'AMBUJACEM.NS', 'ANANDRATHI.NS', 'ANGELONE.NS', 'APARINDS.NS', 'APLAPOLLO.NS', 'APLLTD.NS', 'APOLLOTYRE.NS', 'ARVINDFASN.NS', 'ASTRAL.NS', 'ASTRAMICRO.NS', 'AUBANK.NS', 'AXISBANK.NS', 'AZAD.NS', 'BAJAJ-AUTO.NS', 'BAJAJCON.NS', 'BANDHANBNK.NS', 'BANKBARODA.NS', 'BANSALWIRE.NS', 'BECTORFOOD.NS', 'BEL.NS', 'BHEL.NS', 'BIKAJI.NS', 'BLUEJET.NS', 'BPCL.NS', 'BRIGADE.NS', 'CANFINHOME.NS', 'CANHLIFE.NS', 'CASTROLIND.NS', 'CELLO.NS', 'CERA.NS', 'CHOLAFIN.NS', 'CIPLA.NS', 'COFORGE.NS', 'CPPLUS.NS', 'CREDITACC.NS', 'CROMPTON.NS', 'CUMMINSIND.NS', 'CYIENT.NS', 'CYIENTDLM.NS', 'DABUR.NS', 'DALBHARAT.NS', 'DELHIVERY.NS', 'DEVYANI.NS', 'DLF.NS', 'EMAMILTD.NS', 'ENRIN.NS', 'ERIS.NS', 'ETERNAL.NS', 'EX

In [ ]:
# ──────────────────────────────────────────
#  OPTIONAL: Retry failed symbols one by one
#  Run this cell only when `failed` is non-empty
# ──────────────────────────────────────────

if not failed:
    print("No failed symbols to retry. 🎉")
else:
    print(f"Retrying {len(failed)} failed symbols individually...")
    retry_inserted = 0
    still_failed   = []

    with engine.begin() as conn:
        for sym in failed:
            if sym in last_dates and pd.notna(last_dates[sym]):
                start = (last_dates[sym] + timedelta(days=1)).strftime("%Y-%m-%d")
            else:
                start = (today - timedelta(days=DEFAULT_LOOKBACK)).strftime("%Y-%m-%d")

            print(f"  {sym}  {start} → {end_s}")
            try:
                df = fetch_batch([sym], start, end_s)
                if df.empty:
                    print("    ⚠ Still empty")
                    still_failed.append(sym)
                    continue

                rows = df.to_dict(orient="records")
                conn.execute(
                    text("""
                        INSERT INTO dailyohlc
                            (symbol, [date], [open], high, low, [close], volume, adjclose)
                        SELECT
                            :symbol, :date, :open, :high, :low, :close, :volume, :adjclose
                        WHERE NOT EXISTS (
                            SELECT 1 FROM dailyohlc
                            WHERE symbol = :symbol AND [date] = :date
                        )
                    """),
                    rows
                )
                retry_inserted += len(rows)
                print(f"    ✅ {len(rows)} rows")

            except Exception as e:
                print(f"    ❌ {e}")
                still_failed.append(sym)

            time.sleep(random.uniform(5, 10))   # slower for single retries

    print(f"\nRetry inserted : {retry_inserted}")
    print(f"Still failed   : {still_failed}")